# Setting up Pytorch

- pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
- python3.10 --version //Python 3.10.19
- updated cuda driver to latest stable version


In [1]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

Torch: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 4050 Laptop GPU


# Fetching DataSet from Kaggle

- https://www.kaggle.com/datasets/phiitm/marvel-cinematic-universe-dialogue-dataset/code

In [2]:
%pwd

'/home/kartik-agarwal/Documents/DialogGPT/ml/research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'/home/kartik-agarwal/Documents/DialogGPT/ml'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from ml.src.DialogGPT.constants import *
from ml.src.DialogGPT.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])



    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [8]:
import os
import urllib.request as request
import zipfile
from ml.src.DialogGPT import logger
from ml.src.DialogGPT.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config



    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-03 13:10:40,030: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-03 13:10:40,031: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-03 13:10:40,032: INFO: common: created directory at: artifacts]
[2026-03-03 13:10:40,032: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-03 13:10:42,068: INFO: 427721346: artifacts/data_ingestion/archive.zip download! with following info: 
Content-Type: application/zip
X-GUploader-UploadID: AGQBYWx_N0zDprmzM1Dfj48_NuCkbL_Ro5ysunbQj80gq2cyPDzF2ar9Rj3P56DenD5VC3QsBjmpzqM
Expires: Tue, 03 Mar 2026 07:40:40 GMT
Date: Tue, 03 Mar 2026 07:40:40 GMT
Cache-Control: private, max-age=0
Last-Modified: Sat, 30 May 2020 08:46:16 GMT
ETag: "0a61a6cd9ebe2d7ae210149a3b5a291c"
x-goog-generation: 1590828376634994
x-goog-metageneration: 1
x-goog-stored-content-encoding: identity
x-goog-stored-content-length: 484898
x-goog-hash: crc32c=m0Fm4w==
x-goog-hash: md5=CmGmzZ6+LXriEBSaO1opHA==
x-goog-storage-